# Module 6 - Session 1: Practical Exercises

**Total Time Estimate:** 90-120 minutes

**Objective:** To solidify the concepts of text preprocessing by applying them in code and thinking critically about their implications.

## Setup

Please use Python with the `nltk` and `spacy` libraries.

```bash
pip install nltk spacy
python -m spacy download en_core_web_sm
```

You will also need to download NLTK data.

```python
import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
```


## Exercise 1: Conceptual Questions

### Foundation

Text preprocessing is not just a cleanup step in NLP. It directly changes meaning, vocabulary size, and what the model can learn from language.

### Build

1. **The "Why"**

Text preprocessing is more complex than numerical preprocessing because language is messy, ambiguous, and context-dependent. Scaling a numeric column usually preserves meaning while changing only magnitude, but text preprocessing can remove, merge, or distort meaning if done carelessly. Decisions such as lowercasing, removing stop words, stemming, and lemmatizing all affect whether the model still captures negation, tense, named entities, or domain-specific language.

2. **Stop Word Dilemma**

A standard stop word list can be dangerous here because it often removes words like `not`, `is`, or `could`, even though those words carry critical meaning in fake-news detection. If `This is not true` becomes just `true`, the sentence flips its meaning. In this task, removing stop words without reviewing the list could erase signals that distinguish denial, uncertainty, or factual correction from misinformation.

3. **Stem vs. Lemma Trade-offs**

I would argue for **lemmatization** if search relevance is the primary product goal, because search quality depends on mapping related word forms to real base words while keeping the results readable and semantically cleaner. Stemming is faster and cheaper at very large scale, but it can produce unnatural roots like `comput` or `runn`, which can hurt matching quality, debugging, and ranking logic. The trade-off is speed versus accuracy: stemming wins on raw throughput, while lemmatization usually wins on cleaner normalization and better user-facing search behavior.

### Result

In NLP, preprocessing is part of the modeling decision itself. A small preprocessing choice can preserve meaning, weaken it, or accidentally destroy it.


## Exercise 2: The Preprocessing Pipeline

This exercise builds a reusable `preprocess_text(raw_text)` function with NLTK. The function tokenizes text, normalizes it, removes noise, and returns a cleaner token list.


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Uncomment these lines the first time you run the notebook.
# nltk.download("punkt")
# nltk.download("stopwords")

stop_words = set(stopwords.words("english"))


def preprocess_text(raw_text):
    # Split mixed forms like "1st" and hyphenated words like "T-shirt".
    normalized_text = re.sub(r"(?<=\\d)(?=[A-Za-z])|(?<=[A-Za-z])(?=\\d)", " ", raw_text)
    normalized_text = re.sub(r"[-/]", " ", normalized_text)

    tokens = word_tokenize(normalized_text)
    tokens = [token.lower() for token in tokens]
    tokens = [token for token in tokens if token.isalpha() and len(token) > 1]
    tokens = [token for token in tokens if token not in stop_words]
    return tokens


In [ ]:
sample_text = "The 1st place winner, Dr. Anya Sharma, couldn't believe her T-shirt's amazing design, which cost over $300!"

processed_tokens = preprocess_text(sample_text)
processed_tokens


### Observation (Markdown Answer)

### Foundation

The function follows the requested pipeline: tokenize, lowercase, keep alphabetic tokens, remove stop words, and return the cleaned list.

### Build

A representative output from this implementation is:

```python
['st', 'place', 'winner', 'dr', 'anya', 'sharma', 'believe', 'shirt', 'amazing', 'design', 'cost']
```

The exact result can vary slightly depending on tokenizer behavior and library version, especially around contractions and hyphenated words.

### Result

Even a simple preprocessing pipeline involves small design choices. Handling mixed tokens like `1st` and `T-shirt` can noticeably change the final output.


## Exercise 3: Challenge Problem - Comparative Analysis

### Foundation

The goal here is to compare how plain token cleaning, stemming, and lemmatization change the vocabulary size of a short article.

### Build

For a self-contained notebook, the article text is included directly below. You can replace it with the first 3-4 paragraphs of any online article if you want to experiment further.

### Result

Run the next cells to generate three cleaned versions of the text and compare their token statistics.


In [ ]:
import spacy
from nltk.stem import PorterStemmer

try:
    nlp = spacy.load("en_core_web_sm")
except OSError as exc:
    raise OSError(
        "SpaCy model 'en_core_web_sm' is not installed. Run: python -m spacy download en_core_web_sm"
    ) from exc

stemmer = PorterStemmer()

article_text = """
The Hubble Space Telescope is a large observatory in orbit around Earth. Since its launch in 1990, it has captured detailed images of distant galaxies, glowing nebulae, and planets in our solar system.

Because Hubble circles above the atmosphere, it avoids much of the distortion that affects telescopes on the ground. That clear view has allowed scientists to measure the age of the universe more accurately and to observe stars forming inside dense clouds of gas.

Astronauts visited Hubble several times to repair instruments and install upgrades. Those missions extended the telescope's life and improved the quality of the data it returned to researchers around the world.

Even as newer observatories join it, Hubble remains one of the most influential scientific tools ever built. Its images continue to support discovery, education, and public curiosity about space.
""".strip()

doc = nlp(article_text)


In [ ]:
lowercase_tokens = [
    token.text.lower()
    for token in doc
    if token.is_alpha and not token.is_stop
]

stemmed_tokens = [stemmer.stem(token) for token in lowercase_tokens]

lemmatized_tokens = [
    token.lemma_.lower()
    for token in doc
    if token.is_alpha and not token.is_stop
]


In [ ]:
def describe_tokens(name, tokens):
    print(name)
    print(f"Total tokens: {len(tokens)}")
    print(f"Unique vocabulary: {len(set(tokens))}")
    print(f"Sample tokens: {tokens[:20]}")
    print()


describe_tokens("Lowercase tokens", lowercase_tokens)
describe_tokens("Stemmed tokens", stemmed_tokens)
describe_tokens("Lemmatized tokens", lemmatized_tokens)


### Analysis (Markdown Answer)

### Foundation

Vocabulary size changes because stemming and lemmatization both collapse multiple surface forms into a smaller set of normalized tokens.

### Build

In most runs, the **stemmed** list produces the smallest vocabulary because stemming is the most aggressive method. It strips endings mechanically, so related words are compressed into short roots even when those roots are not clean dictionary forms. **Lemmatization** usually reduces vocabulary too, but it does so more carefully by mapping words to meaningful base forms. The plain **lowercase token** list keeps the most variation because it removes stop words and punctuation but does not normalize word forms beyond casing.

### Result

The **lemmatized** list is usually the most readable and human-like, while the **stemmed** list is usually the most compact. This is the core preprocessing trade-off: stronger normalization reduces vocabulary more, but it can also make the text less natural.
